In [36]:
%load_ext autoreload
%autoreload 2 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from src.config import PROCESSED_DATA_DIR
from src.feature_selection.filter import corr_screening

import numpy as np
import pandas as pd

2026-04-28 17:49:18.011 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: /Users/spencervenancio/Downloads/STAT 479/SatHealth


In [3]:
df = pd.read_csv(PROCESSED_DATA_DIR / "dataset.csv")
X = df.drop(columns=["depr_prev", "NDVI", "NDVI_binary", "NO2_column_number_density"])
y = df["depr_prev"]  

In [4]:
len(corr_screening(df, selection_criterion="p_value", alpha=0.05))


/Users/spencervenancio/Downloads/STAT 479/SatHealth/src/feature_selection/filter.py:35: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corrs[j], pvals[j] = stats.pearsonr(X[:, j], y)


21

In [26]:
def get_sorted_correlations(df: pd.DataFrame, method: str = "pearson", threshold: float = 0.0) -> pd.DataFrame:
    """
    Returns a sorted DataFrame of feature pair correlations (absolute value descending),
    excluding self-correlations and duplicate pairs.

    Args:
        df:        DataFrame of features (numeric columns only).
        method:    Correlation method: 'pearson', 'spearman', or 'kendall'.
        threshold: Only include pairs with |correlation| >= threshold.

    Returns:
        DataFrame with columns ['feature_1', 'feature_2', 'correlation', 'abs_correlation'],
        sorted by abs_correlation descending.
    """
    corr_matrix = df.corr(method=method)

    # Extract upper triangle (excludes self-correlations and duplicates)
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Stack into long-form Series, drop NaNs
    corr_series = upper.stack().reset_index()
    corr_series.columns = ["feature_1", "feature_2", "correlation"]

    corr_series["abs_correlation"] = corr_series["correlation"].abs()

    # Filter by threshold and sort
    result = (
        corr_series[corr_series["abs_correlation"] >= threshold]
        .sort_values("abs_correlation", ascending=False)
        .reset_index(drop=True)
    )

    return result

results = get_sorted_correlations(df, method="pearson", threshold=0.0)
print(results.to_string(index=False))

                                                feature_1                                                 feature_2  correlation  abs_correlation
                             surface_latent_heat_flux_sum                                     total_evaporation_sum     0.999932         0.999932
                                  HHRenter_Occupied_score                                     pctHH_Renter_Occupied     0.993204         0.993204
                                                SDI_score                                                       sdi     0.992898         0.992898
                                 soil_temperature_level_1                                  soil_temperature_level_3     0.990459         0.990459
                                Education_LT12years_score                                   pct_Education_LT12years     0.990314         0.990314
                                  Single_Parent_Fam_score                                     pct_Single_Parent_Fam     0.98